# Build the face training set on Colab (GPU)

Runtime: **Runtime → Change runtime type → T4 GPU**.

Drive layout expected (`MyDrive/faceid/`):
```
faceid/
  faceid-adapter-code.zip           # the code (git archive)
  models/AuraFace-v1/               # AuraFace pack (parent folder must be named 'models')
  raw/ffhq512/*.parquet             # 54 shards, 27.1 GB
  raw/celebamask/CelebAMask-HQ.zip  # 3.15 GB
```
Output: `MyDrive/faceid/faces_built.zip` (~9 GB): unzip it as `data/faces` on the GPU box.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/faceid'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
# Copy everything to the local disk first: reading many files from Drive is slow.
!mkdir -p /content/work/models /content/work/raw
!unzip -q -o "$D/faceid-adapter-code.zip" -d /content/work/code
!cp -r "$D/models/AuraFace-v1" /content/work/models/
!cp -r "$D/raw/ffhq512" /content/work/raw/
!unzip -q -o "$D/raw/celebamask/CelebAMask-HQ.zip" "CelebAMask-HQ/CelebA-HQ-img/*" -d /content/work/raw/
!ls /content/work/raw/ffhq512 | grep -c parquet; ls /content/work/raw/CelebAMask-HQ/CelebA-HQ-img | wc -l

In [ ]:
# insightface (AuraFace runtime) + AuraFace on CUDA + parquet reader
!pip -q install insightface==0.7.3 onnxruntime-gpu pyarrow pyyaml
import onnxruntime as ort
print(ort.get_available_providers())   # must include CUDAExecutionProvider

In [ ]:
# The build. Resumable: re-running skips finished images (as long as /content/work/faces survives).
%cd /content/work/code
!python prep/build_dataset.py --gpu \
    --aura /content/work/models/AuraFace-v1 \
    --parquet "/content/work/raw/ffhq512/*.parquet" \
    --images /content/work/raw/CelebAMask-HQ/CelebA-HQ-img \
    --out /content/work/faces

In [ ]:
# Check, then save ONE zip to Drive (100k small files straight to Drive is very slow)
import json, numpy as np
meta = [json.loads(l) for l in open('/content/work/faces/meta.jsonl')]
emb = np.load('/content/work/faces/embeds.npy')
print(len(meta), 'faces |', emb.shape, '| val', sum(m['split'] == 'val' for m in meta))
assert len(meta) == len(emb)
!cd /content/work && zip -q -r -0 faces_built.zip faces && cp faces_built.zip "$D/" && ls -la "$D/faces_built.zip"